In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)

In [0]:

df = (df
        .dropDuplicates()
        .fillna({"Item_Weight":0, "Outlet_Size": "Unknown"})
        .withColumn("Item_Fat_Content",
                     when(col("Item_Fat_Content").isin("LF", "low fat"),"Low Fat")
                    .when(col("Item_Fat_Content").isin("reg"), "Regular")
                    .otherwise(col("Item_Fat_Content"))
                    )
        
    )

print("Cleaned Rows are:", df.count())
df.display()

**  3. For each Outlet_Identifier, calculates:
Total Sales,
Average Sales,
Number of Unique Items Sold,
Total Sales from High MRP items only (Item_MRP >= 150) 
**

In [0]:
outlet_Summary = (
    df.groupBy("Outlet_Identifier")
    .agg(
        round(sum("Item_Outlet_Sales"), 2).alias("Total Sales"),
        round(avg("Item_Outlet_Sales"), 2).alias("Average Sales"),
        countDistinct("Item_Identifier").alias("Unique Items Sold"),
        round(sum(when(col("Item_MRP")>=150,col("Item_Outlet_Sales")).otherwise(0)), 2).alias("High MRP Sales")

    )
)
outlet_Summary.display()

**STEP 3: Rank outlets by Total Sales (dense rank)**


In [0]:
rank_window = Window.orderBy(desc("Total Sales"))
outlet_ranked = outlet_Summary.withColumn("Outlet Rank", dense_rank().over(rank_window))
outlet_ranked.display()

**Top Selling Items By Outlet**

In [0]:

item_outlet_agg = (
    df.groupBy("Outlet_Identifier", "Item_Type")
    .agg(round(sum("Item_Outlet_Sales"), 2).alias("Item Type Sales"))
)

top_item_window = Window.partitionBy("Outlet_Identifier").orderBy(desc("Item Type Sales"))
top_item_per_outlet = (item_outlet_agg
    .withColumn("Item_Rank", row_number().over(top_item_window))
    .filter(col("Item_Rank") == 1)
    .select("Outlet_Identifier", "Item_Type", "Item Type Sales")
    .withColumnRenamed("Item_Type", "Top_Item_Type")
    .withColumnRenamed("Item Type Sales", "Top_Item_Sales")
)
print("STEP 4 - Top-selling Item_Type per outlet identified")
top_item_per_outlet.display()


**Join**

In [0]:
Final_report = (
    outlet_ranked.join(top_item_per_outlet, on="Outlet_Identifier", how="left")
    .orderBy("Outlet Rank")
     .select(
        "Outlet Rank", "Outlet_Identifier", "Total Sales", "Average Sales",
        "Unique Items Sold", "High MRP Sales", "Top_Item_Type", "Top_Item_Sales"
    )
)

Final_report.display()


**Total Sales percentage**

In [0]:
final_report = (Final_report
                .withColumn("Item_Sales_Percentage", 
                    round(col("Top_Item_Sales")/ col("Total Sales")* 100, 2)
                    )
)
final_report.display()
df.display()